# PHQ-8 Prediction With Frozen Wav2Vec2 Features

This notebook is a first `wav2vec` attempt built around a **frozen pretrained speech encoder**.

**Why this workflow?**
- The dataset is not huge, so fully fine-tuning a large speech model can overfit quickly.
- A frozen encoder gives us strong pretrained speech representations with much less training instability.
- Once embeddings are extracted, we can compare multiple classical regressors in a shared pipeline.

**Pipeline**
1. Load participant-only WAV files from `processed/isolated_audio/`.
2. Map each WAV to participant labels and official split membership.
3. Segment each isolated WAV into fixed windows.
4. Run each window through a pretrained `wav2vec2` encoder.
5. Pool segment embeddings to participant-level features.
6. Train several regressors on the participant-level features.
7. Save fitted models under `best_model/<name>/`.

**First-pass design choice**
- This notebook freezes `facebook/wav2vec2-base-960h` and trains only the regression head.
- If this looks promising later, the next step would be light fine-tuning or trying stronger checkpoints such as `WavLM` or `HuBERT`.


## 1. Install Dependencies

Run this cell if the notebook kernel does not already have the required packages.
The Hugging Face model download requires internet access the first time it is run.


In [ ]:
%pip install -q numpy pandas scikit-learn openpyxl joblib soundfile scipy torch transformers tqdm matplotlib

## 2. Imports And Configuration

In [2]:
from __future__ import annotations

import json
from abc import ABC, abstractmethod
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from tqdm.auto import tqdm
from transformers import AutoProcessor, Wav2Vec2Model

BASE_DIR = Path('.')
ISOLATED_AUDIO_DIR = BASE_DIR / 'processed' / 'isolated_audio'
METADATA_XLSX = BASE_DIR / 'processed' / 'spectrograms' / 'segment_metadata.xlsx'
FEATURE_CACHE_DIR = BASE_DIR / 'processed' / 'wav2vec_features'
MODEL_ROOT = BASE_DIR / 'best_model'

HF_MODEL_NAME = 'facebook/wav2vec2-base-960h'
TARGET_SR = 16_000
SEGMENT_SEC = 8.0
HOP_SEC = 4.0
BATCH_SIZE = 4
RANDOM_STATE = 42
SPLITS = ('train', 'dev', 'test')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Audio dir      : {ISOLATED_AUDIO_DIR.resolve()}')
print(f'Metadata file  : {METADATA_XLSX.resolve()}')
print(f'Feature cache  : {FEATURE_CACHE_DIR.resolve()}')
print(f'Model output   : {MODEL_ROOT.resolve()}')
print(f'Encoder        : {HF_MODEL_NAME}')
print(f'Device         : {DEVICE}')

if not ISOLATED_AUDIO_DIR.exists():
    raise FileNotFoundError(f'Isolated audio directory not found: {ISOLATED_AUDIO_DIR}')
if not METADATA_XLSX.exists():
    raise FileNotFoundError(f'Metadata spreadsheet not found: {METADATA_XLSX}')

Audio dir      : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/isolated_audio
Metadata file  : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/spectrograms/segment_metadata.xlsx
Feature cache  : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/wav2vec_features
Model output   : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model
Encoder        : facebook/wav2vec2-base-960h
Device         : cuda


## 3. Build Participant Table

The spreadsheet already contains participant IDs, PHQ-8 labels, and official split membership. We collapse it to one row per participant and pair each participant with the isolated WAV file.


In [3]:
metadata = pd.read_excel(METADATA_XLSX)
for col in ['participant_id', 'phq_score', 'binary_label', 'participant_dur_s', 'n_turns']:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')
metadata['participant_id'] = metadata['participant_id'].astype(int)
metadata['split'] = metadata['split'].astype(str).str.strip().str.lower()

participant_df = (
    metadata.sort_values(['participant_id', 'segment_idx'])
    .groupby('participant_id', as_index=False)
    .agg({
        'split': 'first',
        'phq_score': 'first',
        'binary_label': 'first',
        'participant_dur_s': 'first',
        'n_turns': 'first',
    })
)

participant_df['session_name'] = participant_df['participant_id'].astype(str) + '_P'
participant_df['audio_path'] = participant_df['session_name'].map(lambda s: ISOLATED_AUDIO_DIR / f'{s}.wav')
participant_df['audio_exists'] = participant_df['audio_path'].map(Path.exists)

missing_audio = participant_df.loc[~participant_df['audio_exists'], ['participant_id', 'audio_path']]
if not missing_audio.empty:
    raise FileNotFoundError(
        'Missing isolated WAV files for some participants. Examples:\n' +
        missing_audio.head(10).to_string(index=False)
    )

display(participant_df.head())
print(participant_df['split'].value_counts().to_string())

,participant_id,split,phq_score,binary_label,participant_dur_s,n_turns,session_name,audio_path,audio_exists
0,300,test,2,0,155.76,87,300_P,processed/isolated_audio/300_P.wav,True
1,301,test,3,0,475.44,104,301_P,processed/isolated_audio/301_P.wav,True
2,302,dev,4,0,208.93,97,302_P,processed/isolated_audio/302_P.wav,True
3,303,train,0,0,642.93,103,303_P,processed/isolated_audio/303_P.wav,True
4,304,train,6,0,362.60,104,304_P,processed/isolated_audio/304_P.wav,True


split
train    107
test      47
dev       35


## 4. Audio Helpers

We keep the raw participant-only waveform, resample to 16 kHz if needed, then cut it into overlapping windows. If a recording is shorter than one window, we keep a single zero-padded segment.


In [4]:
def load_audio_mono(path: Path, target_sr: int = TARGET_SR) -> np.ndarray:
    audio, sr = sf.read(path, always_2d=False)
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        audio = resample_poly(audio, target_sr, sr).astype(np.float32)
    return audio

def segment_audio(audio: np.ndarray, sample_rate: int, segment_sec: float, hop_sec: float) -> np.ndarray:
    segment_len = int(round(segment_sec * sample_rate))
    hop_len = int(round(hop_sec * sample_rate))
    if segment_len <= 0 or hop_len <= 0:
        raise ValueError('segment_sec and hop_sec must be positive')

    if len(audio) == 0:
        return np.zeros((1, segment_len), dtype=np.float32)

    segments = []
    start = 0
    while start + segment_len <= len(audio):
        segments.append(audio[start:start + segment_len])
        start += hop_len

    if not segments:
        padded = np.zeros(segment_len, dtype=np.float32)
        padded[:len(audio)] = audio
        segments.append(padded)

    return np.stack(segments).astype(np.float32)

sample_audio = load_audio_mono(participant_df.iloc[0]['audio_path'])
sample_segments = segment_audio(sample_audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
print('Sample audio length (sec):', len(sample_audio) / TARGET_SR)
print('Segment batch shape      :', sample_segments.shape)

Sample audio length (sec): 155.76
Segment batch shape      : (37, 128000)


## 5. Load Frozen Wav2Vec2 Encoder

This notebook uses the pretrained encoder as a feature extractor. We do not update its weights in this first attempt.


In [5]:
processor = AutoProcessor.from_pretrained(HF_MODEL_NAME)
encoder = Wav2Vec2Model.from_pretrained(HF_MODEL_NAME).to(DEVICE)
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False

encoder_hidden_size = encoder.config.hidden_size
print('Encoder hidden size:', encoder_hidden_size)

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Encoder hidden size: 768


## 6. Extract Segment Embeddings

Each 8-second segment is encoded with `wav2vec2`, then pooled over time with an attention-mask-aware mean. This produces one embedding per segment.


In [6]:
def masked_mean(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    valid_lengths = attention_mask.sum(dim=1).clamp(min=1)
    mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
    summed = (hidden_states * mask).sum(dim=1)
    return summed / valid_lengths.unsqueeze(-1)

def encode_segments(segments: np.ndarray, batch_size: int = BATCH_SIZE) -> np.ndarray:
    embeddings = []
    for start in range(0, len(segments), batch_size):
        batch = segments[start:start + batch_size]
        inputs = processor(
            batch.tolist(),
            sampling_rate=TARGET_SR,
            return_tensors='pt',
            padding=True,
            return_attention_mask=True,
        )
        input_values = inputs['input_values'].to(DEVICE)
        attention_mask = inputs.get('attention_mask')
        if attention_mask is None:
            # Our segments are fixed-length, so an all-ones mask is a safe fallback.
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
        else:
            attention_mask = attention_mask.to(DEVICE)

        with torch.no_grad():
            outputs = encoder(input_values=input_values, attention_mask=attention_mask)
            frame_mask = encoder._get_feature_vector_attention_mask(outputs.last_hidden_state.shape[1], attention_mask)
            pooled = masked_mean(outputs.last_hidden_state, frame_mask)

        embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings).astype(np.float32)

sample_embeddings = encode_segments(sample_segments[:min(len(sample_segments), 2)])
print('Sample embedding shape:', sample_embeddings.shape)

Sample embedding shape: (2, 768)


## 7. Aggregate To Participant-Level Features

The target label is at the participant level, so we pool segment embeddings per participant. This keeps evaluation aligned with the real task and avoids treating every segment as an independent label.

For each participant, we concatenate:
- mean of segment embeddings
- standard deviation of segment embeddings
- max of segment embeddings
- simple metadata: number of segments, isolated duration, number of turns


In [8]:
cache_stub = HF_MODEL_NAME.replace('/', '__')
cache_dir = FEATURE_CACHE_DIR / cache_stub
cache_dir.mkdir(parents=True, exist_ok=True)

participant_cache_dir = cache_dir / 'participant_chunks'
participant_cache_dir.mkdir(parents=True, exist_ok=True)

participant_feature_rows = []

for row in tqdm(participant_df.itertuples(index=False), total=len(participant_df), desc='Encoding participants'):
    cache_file = participant_cache_dir / f'{int(row.participant_id)}.npz'

    if cache_file.exists():
        cached = np.load(cache_file, allow_pickle=True)
        feature_vector = cached['feature_vector'].astype(np.float32)
        n_segments = int(cached['n_segments'])
    else:
        audio = load_audio_mono(row.audio_path)
        segments = segment_audio(audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
        segment_embeddings = encode_segments(segments)

        feature_vector = np.concatenate([
            segment_embeddings.mean(axis=0),
            segment_embeddings.std(axis=0),
            segment_embeddings.max(axis=0),
            np.array([
                len(segments),
                len(audio) / TARGET_SR,
                float(row.n_turns),
            ], dtype=np.float32),
        ]).astype(np.float32)

        n_segments = int(len(segments))

        np.savez_compressed(
            cache_file,
            participant_id=int(row.participant_id),
            session_name=str(row.session_name),
            split=str(row.split),
            phq_score=float(row.phq_score),
            binary_label=int(row.binary_label),
            n_segments=n_segments,
            feature_vector=feature_vector,
        )

    participant_feature_rows.append({
        'participant_id': int(row.participant_id),
        'session_name': str(row.session_name),
        'split': str(row.split),
        'phq_score': float(row.phq_score),
        'binary_label': int(row.binary_label),
        'n_segments': n_segments,
        'feature_vector': feature_vector,
    })

feature_df = pd.DataFrame(participant_feature_rows).sort_values('participant_id').reset_index(drop=True)
X = np.vstack(feature_df['feature_vector'].to_list())

np.save(cache_dir / 'participant_features.npy', X)
feature_df.drop(columns=['feature_vector']).to_csv(cache_dir / 'participant_metadata.csv', index=False)

done_ids = sorted(int(p.stem) for p in participant_cache_dir.glob('*.npz'))
print('Participant feature matrix:', X.shape)
print(f'Saved cached features to: {cache_dir.resolve()}')
print(f'Cached participants: {len(done_ids)} / {len(participant_df)}')
print('Last cached participant IDs:', done_ids[-10:] if done_ids else [])


Encoding participants:   0%|          | 0/189 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Split Features Into Train, Dev, And Test

In [ ]:
split_data = {}
split_ids = {}

for split_name in SPLITS:
    subset = feature_df[feature_df['split'] == split_name].reset_index(drop=True)
    X_split = np.vstack(subset['feature_vector'].to_list())
    y_split = subset['phq_score'].to_numpy(dtype=np.float32)
    split_data[split_name] = (X_split, y_split)
    split_ids[split_name] = subset['participant_id'].to_numpy(dtype=int)
    print(f'{split_name:5s}: X={X_split.shape}, y={y_split.shape}, mean PHQ={y_split.mean():.2f}')

## 9. Shared Regressor Interface

The encoder is frozen, so the trainable part becomes a classical regressor. We keep the same wrapper pattern across models so train/dev/test evaluation stays parallel and easy to compare.


In [ ]:
def pearson_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size < 2:
        return 0.0
    if np.std(y_true) == 0.0 or np.std(y_pred) == 0.0:
        return 0.0
    return float(np.corrcoef(y_true, y_pred)[0, 1])

class BaseRegressor(ABC):
    def __init__(self, name: str):
        self.name = name
        self.pipeline = self.build_pipeline()

    @abstractmethod
    def build_pipeline(self) -> Pipeline:
        raise NotImplementedError

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.pipeline.fit(X, y)
        return self

    def forward(self, X: np.ndarray) -> np.ndarray:
        return self.pipeline.predict(X)

    def loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = mean_absolute_error(y_true, y_pred)
        r = pearson_correlation(y_true, y_pred)
        return {
            'mse': float(mse),
            'rmse': float(rmse),
            'mae': float(mae),
            'pearson_r': float(r),
        }

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, dict]:
        preds = self.forward(X)
        metrics = self.loss(y, preds)
        return preds, metrics

    def save(self, output_dir: Path, metrics: dict, predictions: dict, participant_ids: dict, config: dict):
        output_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.pipeline, output_dir / 'model.joblib')
        with (output_dir / 'metrics.json').open('w', encoding='utf-8') as f:
            json.dump(metrics, f, indent=2)
        with (output_dir / 'config.json').open('w', encoding='utf-8') as f:
            json.dump(config, f, indent=2)

        rows = []
        for split_name, split_preds in predictions.items():
            for pid, pred in zip(participant_ids[split_name], split_preds):
                rows.append({
                    'split': split_name,
                    'participant_id': int(pid),
                    'predicted_phq_score': float(pred),
                })
        pd.DataFrame(rows).to_csv(output_dir / 'predictions.csv', index=False)

class RidgeRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', Ridge(alpha=10.0)),
        ])

class ElasticNetRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', ElasticNet(alpha=0.03, l1_ratio=0.3, max_iter=10000, random_state=RANDOM_STATE)),
        ])

class SVRRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', SVR(kernel='rbf', C=10.0, epsilon=0.5)),
        ])

class RandomForestBaseline(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('model', RandomForestRegressor(
                n_estimators=400,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

models = {
    'wav2vec2_ridge': RidgeRegressor('wav2vec2_ridge'),
    'wav2vec2_elastic_net': ElasticNetRegressor('wav2vec2_elastic_net'),
    'wav2vec2_svr': SVRRegressor('wav2vec2_svr'),
    'wav2vec2_random_forest': RandomForestBaseline('wav2vec2_random_forest'),
}

print('Models:', ', '.join(models.keys()))

## 10. Train, Evaluate, And Save Models

Each model is trained once on the participant-level train split. We report train/dev/test metrics in the same style and save the fitted pipeline plus predictions under `best_model/<model-name>/`.


In [ ]:
X_train, y_train = split_data['train']

results = []
run_config = {
    'encoder_name': HF_MODEL_NAME,
    'target_sample_rate': TARGET_SR,
    'segment_sec': SEGMENT_SEC,
    'hop_sec': HOP_SEC,
    'batch_size': BATCH_SIZE,
    'feature_layout': ['segment_mean', 'segment_std', 'segment_max', 'n_segments', 'isolated_duration', 'n_turns'],
}

for model_name, model in models.items():
    print(f'\n=== Training {model_name} ===')
    model.fit(X_train, y_train)

    split_metrics = {}
    split_predictions = {}

    for split_name in SPLITS:
        X_split, y_split = split_data[split_name]
        preds, metrics = model.evaluate(X_split, y_split)
        split_metrics[split_name] = metrics
        split_predictions[split_name] = preds

        results.append({
            'model': model_name,
            'split': split_name,
            **metrics,
        })

        print(
            f"{split_name:5s} | MSE={metrics['mse']:.4f} | RMSE={metrics['rmse']:.4f} "
            f"| MAE={metrics['mae']:.4f} | r={metrics['pearson_r']:.4f}"
        )

    output_dir = MODEL_ROOT / model_name
    model.save(output_dir, split_metrics, split_predictions, split_ids, run_config)
    print(f'Saved to: {output_dir.resolve()}')

results_df = pd.DataFrame(results)
results_df

## 11. Compare Models

In [ ]:
summary = results_df.pivot(index='model', columns='split', values=['rmse', 'mae', 'pearson_r'])
summary = summary.sort_values(('rmse', 'dev'))
summary

## 12. Thoughts On Interpreting The Results

When this notebook finishes, here is how to read the outcome:
- If `wav2vec2_*` models noticeably beat the spectrogram-summary baselines on dev RMSE and Pearson correlation, the pretrained speech representation is adding value.
- If the gains are small but consistent, the next step is richer pooling or a stronger downstream head.
- If the gains are large, that is a good sign to try light fine-tuning or stronger speech checkpoints later.
- If performance is still poor, the bottleneck may be label noise, limited data, or the need for more task-specific temporal modeling.
